# 00 — Project Setup (FIXED)

Initializes the repository and performs explicit dependency/folder checks.

**Important changes**
- Does not globally suppress warnings.
- Keeps raw data immutable.
- Defines the study design used by the corrected workflow.
- `0.005°` is treated as the **output grid spacing**, not a claim that all rainfall information truly has 500 m native resolution.

In [1]:
from pathlib import Path
import warnings

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists():
            return candidate
    raise FileNotFoundError(
        "Project root not found. Run this notebook from inside the repository."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"

for d in [INTERIM_DIR, PROCESSED_DIR, OUTPUT_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)

PROJECT_ROOT = E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh


In [2]:
import importlib.util, sys

required = ["numpy", "pandas", "rasterio", "pyproj", "sklearn", "geopandas", "matplotlib"]
optional = ["xgboost", "lightgbm", "catboost", "yaml"]

print("Python:", sys.version.split()[0])
print("\nRequired packages")
missing = []
for p in required:
    ok = importlib.util.find_spec(p) is not None
    print(f"{p:12s}: {'OK' if ok else 'MISSING'}")
    if not ok:
        missing.append(p)

print("\nOptional packages")
for p in optional:
    print(f"{p:12s}: {'OK' if importlib.util.find_spec(p) else 'MISSING'}")

if missing:
    raise ImportError("Install missing required packages: " + ", ".join(missing))

Python: 3.11.11

Required packages
numpy       : OK
pandas      : OK
rasterio    : OK
pyproj      : OK
sklearn     : OK
geopandas   : OK
matplotlib  : OK

Optional packages
xgboost     : OK
lightgbm    : OK
catboost    : OK
yaml        : OK


In [3]:
# ---------------- STUDY DESIGN ----------------
TRAIN_YEARS = [2017, 2018, 2019, 2020]
VALID_YEARS = [2021]
TEST_YEARS  = [2022]

# Base-paper precipitation products, mapped to your folder names.
PRECIP_PRODUCTS = [
    "CCS",
    "PDIR",
    "GSMaP_MVK",
    "CDR",          # PERSIANN-CDR
    "CHIRPS",
    "IMERG",
    "GSMaP_Gauge_v7",
    "ERA5",
]

# Adapted land predictors. LST Night is intentionally NOT used.
LAND_FEATURES = ["DEM", "NDVI", "LST_Day", "Distance_Sea"]

# This is an output grid spacing, not an assertion of effective information resolution.
TARGET_RES_DEG = 0.005
DST_NODATA = -9999.0

print("Train:", TRAIN_YEARS)
print("Validation:", VALID_YEARS)
print("Independent test:", TEST_YEARS)
print("Precipitation products:", PRECIP_PRODUCTS)
print("Land features:", LAND_FEATURES)
print("Output grid spacing:", TARGET_RES_DEG, "degree")

Train: [2017, 2018, 2019, 2020]
Validation: [2021]
Independent test: [2022]
Precipitation products: ['CCS', 'PDIR', 'GSMaP_MVK', 'CDR', 'CHIRPS', 'IMERG', 'GSMaP_Gauge_v7', 'ERA5']
Land features: ['DEM', 'NDVI', 'LST_Day', 'Distance_Sea']
Output grid spacing: 0.005 degree


In [4]:
required_folders = [
    RAW_DIR / "boundary",
    RAW_DIR / "gauge",
    RAW_DIR / "precipitation",
    RAW_DIR / "predictors",
]
for d in required_folders:
    if not d.exists():
        raise FileNotFoundError(f"Missing required folder: {d}")
    print("[OK]", d.relative_to(PROJECT_ROOT))

print("\nSetup complete.")

[OK] data\raw\boundary
[OK] data\raw\gauge
[OK] data\raw\precipitation
[OK] data\raw\predictors

Setup complete.
